#<font color="blue"><b>******* Supervised Fine-Tuning *******</b></font>

### <font color="blue"><b> Import Required Libraries

- <font color="black"><b>**torch** – Provides tensor operations and deep learning functionality.
- **Dataset** – Creates a custom PyTorch dataset.
- **load_dataset** – Downloads datasets from the Hugging Face Hub.
- **transformers** – Provides pretrained language models, tokenizers, and training utilities.

In [ ]:
import random
import torch
from torch.utils.data import Dataset
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

### <font color="blue"><b>Configuration

- **MODEL_NAME** specifies the pretrained language model.
- **MAX_LEN** defines the maximum sequence length after tokenization.
- **IGNORE_INDEX** specifies the label value ignored during loss computation so that prompt tokens do not contribute to training.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
DATASET = "databricks/databricks-dolly-15k"
MAX_LEN = 256
IGNORE_INDEX = -100

### <font color="blue"><b> Load the Pretrained Model and Tokenizer

The tokenizer converts text into numerical token IDs, while the pretrained language model initializes with weights learned during pretraining. Since the selected Qwen model does not define a padding token, the end-of-sequence (EOS) token is used as the padding token.
Reference link - https://github.com/QwenLM/Qwen/blob/main/tokenization_note.md

In [ ]:
# ---------------------------------------------------------------------------
# Load model and tokenizer
# ---------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

### <font color="blue"><b> Encode Instruction(input)–Response(output) Pairs

Each training example consists of a **user instruction** and its corresponding **assistant response**.

This function performs the following steps:

1. Creates the user prompt.
2. Builds the complete conversation.
3. Applies the Qwen chat template.
4. Converts text into token IDs.
5. Masks prompt tokens using `IGNORE_INDEX`.
6. Uses only assistant tokens as prediction targets.
7. Pads sequences to a fixed length.
8. Creates an attention mask for the model.

In [ ]:
# ---------------------------------------------------------------------------
# Turn one (input, target) pair into token ids + labels
# ---------------------------------------------------------------------------
def encode_example(user_msg, assistant_msg):
    messages_prompt_only = [{"role": "user", "content": user_msg}]
    messages_full = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]
    prompt_text = tokenizer.apply_chat_template(messages_prompt_only, tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(messages_full, tokenize=False, add_generation_prompt=False)

    encoding = tokenizer(full_text, add_special_tokens=False, return_offsets_mapping=True)
    full_ids = encoding["input_ids"]
    offsets = encoding["offset_mapping"]  # list of (char_start, char_end) per token, relative to full_text
    # Truncate to max length first
    input_ids = full_ids[:MAX_LEN]
    offsets = offsets[:MAX_LEN]
    prompt_char_len = len(prompt_text)

    prompt_len = len(input_ids)  # default: if nothing crosses the boundary, mask nothing extra
    for i, (start, end) in enumerate(offsets):
        if start >= prompt_char_len:
            prompt_len = i
            break

    labels = []
    for i in range(len(input_ids)):
        if i < prompt_len:
            labels.append(IGNORE_INDEX)   # Mask out user prompt tokens
        else:
            labels.append(input_ids[i])   # Loss on assistant response tokens

    # Keep track of the actual content length before applying padding
    actual_len = len(input_ids)

    # Pad sequences cleanly up to MAX_LEN
    while len(input_ids) < MAX_LEN:
        input_ids.append(tokenizer.pad_token_id)
        labels.append(IGNORE_INDEX)       # Padding tokens must always be ignored

    # Construct the attention mask using the content length.
    attention_mask = [1] * actual_len + [0] * (MAX_LEN - actual_len)

    return {
        "input_ids": torch.tensor(input_ids),
        "attention_mask": torch.tensor(attention_mask),
        "labels": torch.tensor(labels),
    }

In [ ]:
encode_example("How's the session?","Nice")

{'input_ids': tensor([151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,   4340,    594,    279,   3797,
             30, 151645,    198, 151644,  77091,    198,  44978, 151645,    198,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643

In [ ]:
# Understanding of chat_template
chat = [
  {"role": "user", "content": "Hello, how are you?"},
  {"role": "assistant", "content": "I'm doing great. How can I help you today?"},
  {"role": "user", "content": "I'd like to show off how chat templating works!"},
]

print("With-------", tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True))
print("--------------------------")
print("Without----", tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=False))

With------- <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hello, how are you?<|im_end|>
<|im_start|>assistant
I'm doing great. How can I help you today?<|im_end|>
<|im_start|>user
I'd like to show off how chat templating works!<|im_end|>
<|im_start|>assistant

--------------------------
Without---- <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hello, how are you?<|im_end|>
<|im_start|>assistant
I'm doing great. How can I help you today?<|im_end|>
<|im_start|>user
I'd like to show off how chat templating works!<|im_end|>



### Understanding generation_prompt

| **add_generation_prompt=True** | **add_generation_prompt=False** |
|--------------------------------|---------------------------------|
| Used when you want the model to **generate a response**. | Used when the **assistant response is already available**. |
| Appends the **assistant start token** (generation prompt). | Does **not append any additional prompt**. |
| Creates an **incomplete conversation** where the assistant response is yet to be generated. | Creates a **complete conversation** containing both the user instruction and assistant response. |

In [ ]:
encode_example("How's you", "I'm fine")

{'input_ids': tensor([151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,   4340,    594,    498, 151645,
            198, 151644,  77091,    198,     40,   2776,   6915, 151645,    198,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643

### <font color="blue"><b> Create a Custom Dataset

A custom PyTorch dataset is created to efficiently load training examples.

Each dataset item contains an instruction-response pair, which is converted into model inputs using the encoding function whenever it is accessed during training.

In [ ]:
# ---------------------------------------------------------------------------
# dataset wrapper
# ---------------------------------------------------------------------------
class SFTDataset(Dataset):
    def __init__(self, examples): # storing examples in the dataset
        self.examples = examples

    def __len__(self):  # number of examples in the dataset
        return len(self.examples)

    def __getitem__(self, idx): # geting one example encoded from the dataset
        example = self.examples[idx]
        return encode_example(example["input"], example["target"])

### <font color="blue"><b> Load the Dolly-15K Dataset

The Databricks Dolly-15K dataset contains thousands of instruction-response examples for instruction-following tasks.

For supervised fine-tuning:

- The **instruction** is used as the model input.
- The **response** is used as the target output.

Each record is converted into the following format:

```python
{
    "input": instruction,
    "target": response
}
```

In [ ]:
# ---------------------------------------------------------------------------
# Load the Dolly-15k dataset
# ---------------------------------------------------------------------------
raw_data = load_dataset(DATASET, split="train")
all_examples = []
for row in raw_data:
    input_text = row["instruction"]
    example = {"input": input_text, "target": row["response"]}
    all_examples.append(example)

random.seed(0)
random.shuffle(all_examples)

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

### <font color="blue"><b> Shuffle and Split the Dataset

Before training, the dataset is randomly shuffled to ensure a balanced distribution of examples.

The shuffled dataset is divided into three subsets:

- **Training Set** – Used to update the model parameters.
- **Validation Set** – Used to evaluate the model after each training epoch.
- **Test Set** – Used only after training for qualitative evaluation.

In [ ]:
# ---------------------------------------------------------------------------
# Split into train / validation / test
# ---------------------------------------------------------------------------
N_TRAIN = 200
N_VAL = 20
N_TEST = 5

train_examples = all_examples[:N_TRAIN]
val_examples = all_examples[N_TRAIN:N_TRAIN + N_VAL]
test_examples = all_examples[N_TRAIN + N_VAL:N_TRAIN + N_VAL + N_TEST]

print("Training examples:", len(train_examples))
print("Validation examples:", len(val_examples))
print("Test examples:", len(test_examples))

train_dataset = SFTDataset(train_examples)
val_dataset = SFTDataset(val_examples)

Training examples: 200
Validation examples: 20
Test examples: 5


### <font color="blue"><b> Configure Training

The `TrainingArguments` class defines the training configuration.

Important settings include:

- Number of training epochs
- Batch size
- Learning rate
- Validation frequency
- Model checkpoint saving
- Logging configuration

These parameters control how the model is optimized during supervised fine-tuning.

### <font color="blue"><b> Initialize the Trainer

The Hugging Face `Trainer` simplifies the fine-tuning process by managing the complete training workflow.

It automatically performs:

- Forward propagation
- Loss computation
- Backpropagation
- Gradient updates
- Validation
- Model checkpoint saving

### <font color="blue"><b> Fine-Tune the Model

The model is trained on the instruction-response dataset.

During training:

- Model parameters are updated using the training set.
- Validation is performed after every epoch.
- Checkpoints are saved after each epoch.
- The final fine-tuned model is stored for later use.

In [ ]:
# ---------------------------------------------------------------------------
# Train (with validation each epoch)
# ---------------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./sft_out",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)
trainer.train()
trainer.save_model("./sft_out")

final_val_metrics = trainer.evaluate()
print("\nFinal validation metrics:", final_val_metrics)

Epoch,Training Loss,Validation Loss
1,2.312992,2.525488
2,1.136444,2.447126
3,0.579489,2.549435


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch
0.579489,2.549435,3



Final validation metrics: {'eval_loss': 2.5494351387023926}


### <font color="blue"><b> Generate Responses

After fine-tuning, the model can generate responses for new user instructions.

This function:

1. Formats the instruction using the Qwen chat template.
2. Tokenizes the prompt.
3. Generates new tokens using autoregressive decoding.
4. Converts the generated tokens back into readable text.

In [ ]:
# ---------------------------------------------------------------------------
# Simple qualitative evaluation on the held-out test set
# ---------------------------------------------------------------------------
def generate_response(user_msg):
    messages = [{"role": "user", "content": user_msg}]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).to(model.device)

    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

### <font color="blue"><b> Evaluate on the Test Set

Finally, the fine-tuned model is evaluated using previously unseen test examples.

For each test sample, the notebook displays:

- The user instruction
- The reference response from the dataset
- The model-generated response

This qualitative evaluation helps assess how well the model has learned the instruction-following task.

In [ ]:
print("\n=== Test Set Evaluation ===")
for example in test_examples:
    prediction = generate_response(example["input"])
    print("\nQuery     :", example["input"])
    print("Reference :", example["target"])
    print("Prediction:", prediction)


=== Test Set Evaluation ===

Query     : What are the Olympic Games?
Reference : The modern Olympic Games or Olympics (French: Jeux olympiques)are the leading international sporting events featuring summer and winter sports competitions in which thousands of athletes from around the world participate in a variety of competitions. The Olympic Games are considered the world's foremost sports competition with more than 200 teams, representing sovereign states and territories, participating. The Olympic Games are normally held every four years, and since 1994, have alternated between the Summer and Winter Olympics every two years during the four-year period.
Prediction: The Olympic Games are an international sporting event that is held every four years. It is the world's largest sports event and is attended by over two and a half million people. It is also the world's third largest economy, after the United States and China. The Olympic Games are named after the Greek god Zeus, who was th